# Hyperparameter Optimization

## CircuitBench Tutorial 6

This notebook demonstrates systematic hyperparameter optimization for surrogate machine learning models using reproducible search strategies and cross-validation.

### Learning Objectives

- Load benchmark datasets
- Define hyperparameter search spaces
- Perform Grid Search
- Perform Random Search
- Compare optimized models
- Analyze optimization results
- Export reproducible reports


## Scientific Background

Hyperparameter optimization is essential for obtaining optimal predictive performance while avoiding overfitting. CircuitBench supports reproducible optimization workflows using cross-validation and standardized evaluation metrics.

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


In [ ]:
import circuitbench
from circuitbench.benchmark import BenchmarkRunner

## Load Benchmark Dataset

In [ ]:
runner = BenchmarkRunner()
benchmark = runner.load('CMOS_Inverter')

## Prepare Training Data

In [ ]:
X_train = benchmark.X_train
y_train = benchmark.y_train

X_test = benchmark.X_test
y_test = benchmark.y_test

## Define Baseline Model

In [ ]:
model = RandomForestRegressor(
    random_state=SEED
)

## Hyperparameter Search Space

In [ ]:
parameter_grid = {
    'n_estimators':[50,100,200,500],
    'max_depth':[None,5,10,20],
    'min_samples_split':[2,5,10],
    'min_samples_leaf':[1,2,4],
    'max_features':['sqrt','log2']
}

parameter_grid

## Grid Search Optimization

In [ ]:
grid_search = GridSearchCV(
    estimator=model,
    param_grid=parameter_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

## Best Grid Search Parameters

In [ ]:
grid_search.best_params_

## Best Grid Search Score

In [ ]:
grid_search.best_score_

## Random Search Optimization

In [ ]:
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=parameter_grid,
    n_iter=25,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

random_search.fit(X_train, y_train)

## Best Random Search Parameters

In [ ]:
random_search.best_params_

## Compare Optimization Strategies

In [ ]:
comparison = pd.DataFrame({
    'Method':['Grid Search','Random Search'],
    'Best Score':[
        grid_search.best_score_,
        random_search.best_score_
    ]
})

comparison

## Save Optimization Summary

In [ ]:
output_dir = Path('hyperparameter_results')
output_dir.mkdir(exist_ok=True)

comparison.to_csv(
    output_dir/'optimization_comparison.csv',
    index=False
)

## Evaluate Optimized Models

In [ ]:
grid_predictions = grid_search.best_estimator_.predict(X_test)
random_predictions = random_search.best_estimator_.predict(X_test)

## Test Set Performance

In [ ]:
results = pd.DataFrame({
    'Method': ['Grid Search', 'Random Search'],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, grid_predictions)),
        np.sqrt(mean_squared_error(y_test, random_predictions))
    ]
})

results

## Optimization Performance Comparison

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(results['Method'], results['RMSE'])
plt.ylabel('RMSE')
plt.title('Hyperparameter Optimization Performance')
plt.tight_layout()

## Cross-Validation Results

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)

cv_results.head()

## Top 10 Hyperparameter Configurations

In [ ]:
top_configs = cv_results.sort_values(
    by='rank_test_score'
).head(10)

top_configs

## Export Cross-Validation Results

In [ ]:
cv_results.to_csv(
    output_dir/'grid_search_cv_results.csv',
    index=False
)

## Export Optimization Report

In [ ]:
report = {
    'best_method': 'Grid Search' if grid_search.best_score_ >= random_search.best_score_ else 'Random Search',
    'grid_best_score': float(grid_search.best_score_),
    'random_best_score': float(random_search.best_score_),
    'grid_best_parameters': grid_search.best_params_,
    'random_best_parameters': random_search.best_params_,
    'random_seed': SEED
}

import json

with open(output_dir/'optimization_report.json', 'w') as f:
    json.dump(report, f, indent=4)


## Reproducibility Checklist

- Fixed random seed used for all optimization experiments.
- Hyperparameter search space fully documented.
- Five-fold cross-validation applied consistently.
- Optimization results exported.
- Best hyperparameters recorded.
- Cross-validation statistics archived.
- Optimization report saved in JSON format.


## Best Practices

1. Evaluate optimized models on an independent test set.
2. Use cross-validation to reduce selection bias.
3. Record all search spaces and optimization settings.
4. Compare multiple optimization strategies.
5. Save optimization artifacts for reproducibility.


## Next Notebook

Continue with **7_SHAP_Explainability.ipynb** to interpret surrogate models using SHAP values, feature attribution, local explanations, and publication-quality explainable AI visualizations.

# Summary

In this notebook we:

- Defined a reproducible hyperparameter search space.
- Optimized a Random Forest surrogate model using Grid Search and Random Search.
- Compared optimization strategies through cross-validation.
- Evaluated optimized models on an independent test set.
- Exported optimization results, cross-validation statistics, and benchmark reports.

These procedures provide a rigorous and reproducible framework for hyperparameter optimization within CircuitBench.